<a href="https://colab.research.google.com/github/AndrijaBilic/FER_Zavrsni_Rad/blob/main/EnumerabilityDirections.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Enumerability Directions in Qwen3-4B

Adaptation of [Lavi et al. 2025 — *Detecting (Un)answerability with Linear Directions*](https://github.com/MaorLavi/unanswerability-directions) for the **enumerability** property.

**Goal:** find a direction in the model's residual stream that separates
- **Label 1 (enumerable):** questions that have multiple valid answers
- **Label 0 (singular):** questions that have exactly one answer

Then use that direction for **projection-based classification** and **activation steering**.

---
### Pipeline overview
1. Load & prepare datasets (WebQuestions, AmbigQA, SituatedQA)
2. Build `Qwen3ModelBase`
3. Generate candidate directions (mean-diff over all layers)
4. Select the best (position, layer) by steering
5. Find classification threshold (ROC curve)
6. Evaluate by projection
7. Activation steering demo

## 0 — Install dependencies

In [3]:
!pip install datasets transformers bitsandbytes accelerate jaxtyping scikit-learn tqdm -q

# Clone the original repo so we can reuse their pipeline utilities
!git clone https://github.com/MaorLavi/unanswerability-directions.git
import sys
sys.path.insert(0, 'unanswerability-directions')

# Copy our Qwen3ModelBase into the cloned repo's model_utils folder
import shutil, os
os.makedirs('unanswerability-directions/pipeline/model_utils', exist_ok=True)


fatal: destination path 'unanswerability-directions' already exists and is not an empty directory.


## 1 — Load datasets and build train/val/test splits

In [4]:
from datasets import load_dataset
import pandas as pd
import numpy as np

# ── WebQuestions ──────────────────────────────────────────────────────
ds_webq = load_dataset("web_questions", split="train")
records = []
for ex in ds_webq:
    n = len(ex["answers"])
    if n >= 3:   label = 1
    elif n == 1: label = 0
    else:        continue
    records.append({"question": ex["question"], "label": label, "source": "webquestions"})
df_webq = pd.DataFrame(records)
print("WebQ:", df_webq["label"].value_counts().to_dict())

# ── AmbigQA ───────────────────────────────────────────────────────────
ds_ambig = load_dataset("ambig_qa", "light", split="train")
records = []
for ex in ds_ambig:
    t = ex["annotations"]["type"][0]
    if t == "multipleQAs":   label = 1
    elif t == "singleAnswer": label = 0
    else:                     continue
    records.append({"question": ex["question"], "label": label, "source": "ambigqa"})
df_ambig = pd.DataFrame(records)
print("AmbigQA:", df_ambig["label"].value_counts().to_dict())

# ── SituatedQA ────────────────────────────────────────────────────────
url_train = "https://raw.githubusercontent.com/mikejqzhang/SituatedQA/master/data/raw_data/temp.train.jsonl"
df_situated_raw = pd.read_json(url_train, lines=True)
records = []
for _, ex in df_situated_raw.iterrows():
    label = 1 if ex["is_dependent"] == True else 0
    records.append({"question": ex["question"], "label": label, "source": "situatedqa"})
df_situated = pd.DataFrame(records)
print("SituatedQA:", df_situated["label"].value_counts().to_dict())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/260k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/142k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3778 [00:00<?, ? examples/s]

...

light/train-00000-of-00001.parquet:   0%|          | 0.00/1.41M [00:00<?, ?B/s]

light/validation-00000-of-00001.parquet:   0%|          | 0.00/367k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10036 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2002 [00:00<?, ? examples/s]

AmbigQA: {0: 5289, 1: 4747}
SituatedQA: {0: 1656, 1: 1248}


In [5]:
from sklearn.model_selection import train_test_split

def split_dataset(df, train_size=0.7, val_size=0.15, seed=42):
    """
    Stratified split → train / val / test.
    Returns lists of question strings (what generate_directions expects).
    """
    df_train, df_tmp = train_test_split(
        df, train_size=train_size, stratify=df["label"], random_state=seed
    )
    rel_val = val_size / (1 - train_size)
    df_val, df_test = train_test_split(
        df_tmp, train_size=rel_val, stratify=df_tmp["label"], random_state=seed
    )
    return df_train, df_val, df_test

def to_lists(df):
    """Return (enumerable_questions, singular_questions) as plain string lists."""
    enum_qs = df[df["label"] == 1]["question"].tolist()
    sing_qs = df[df["label"] == 0]["question"].tolist()
    return enum_qs, sing_qs

# Choose your working dataset — change this cell to switch datasets
DATASET_NAME = "ambigqa"   # "webquestions" | "ambigqa" | "situatedqa"
DATASET_MAP  = {"webquestions": df_webq, "ambigqa": df_ambig, "situatedqa": df_situated}

df_work = DATASET_MAP[DATASET_NAME]
df_train, df_val, df_test = split_dataset(df_work)

enum_train, sing_train = to_lists(df_train)
enum_val,   sing_val   = to_lists(df_val)
enum_test,  sing_test  = to_lists(df_test)

print(f"Dataset: {DATASET_NAME}")
print(f"  Train — enum: {len(enum_train)}, singular: {len(sing_train)}")
print(f"  Val   — enum: {len(enum_val)},   singular: {len(sing_val)}")
print(f"  Test  — enum: {len(enum_test)},  singular: {len(sing_test)}")

Dataset: ambigqa
  Train — enum: 3323, singular: 3702
  Val   — enum: 712,   singular: 793
  Test  — enum: 712,  singular: 794


## 2 — Load Qwen3-4B via our ModelBase

In [15]:
from pipeline.model_utils.qwen3_model import Qwen3Model
from pipeline.model_utils.model_factory import construct_model_base

# The factory will now handle the load_in_4bit kwarg via self.loading_kwargs
model_base = construct_model_base("Qwen/Qwen3-4B", load_in_4bit=True)

n_layers = model_base.model.config.num_hidden_layers
d_model  = model_base.model.config.hidden_size
print(f"Model loaded. Layers: {n_layers}, d_model: {d_model}")
print(f"EOI tokens: {model_base.eoi_toks}")
print(f"  decoded: {[model_base.tokenizer.decode([t]) for t in model_base.eoi_toks]}")

TypeError: construct_model_base() got an unexpected keyword argument 'load_in_4bit'

## 3 — Generate candidate directions

For every **(position, layer)** pair, compute:
```
direction[pos, layer] = mean_hidden(enumerable) − mean_hidden(singular)
```
This is the paper's `generate_directions` verbatim — no changes needed.

In [ ]:
from pipeline.generate_directions import generate_directions
import os

ARTIFACT_DIR = f"runs/{DATASET_NAME}"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# generate_directions(model_base, class_A, class_B, ...)
# We map:  class_A = enumerable  (paper's "unanswerable")
#          class_B = singular    (paper's "answerable")
# So the direction points from singular → enumerable
candidate_directions = generate_directions(
    model_base,
    enum_train,    # "positive" class (enumerable)
    sing_train,    # "negative" class (singular)
    ARTIFACT_DIR,
    batch_size=4,  # keep small for Colab
)

print(f"candidate_directions shape: {candidate_directions.shape}")
# Expected: (n_eoi_positions, n_layers, d_model)

## 4 — Select the best direction via activation steering

For each (position, layer) candidate we **add** the direction to the
residual stream and check whether the model's output changes in the
expected way: adding direction should push singular → enumerable behavior,
subtracting should push enumerable → singular.

We operationalise this by looking at how the model *completes* a question:
an enumerable question typically gets a list-like response; a singular one
gets a single entity. We score by whether completions on the val set
change appropriately after the addition.

The paper's `select_direction` does this automatically.

In [ ]:
from pipeline.select_by_steering import select_direction

SELECT_DIR = f"{ARTIFACT_DIR}/select_by_steering"
os.makedirs(SELECT_DIR, exist_ok=True)

pos, layer, best_dir = select_direction(
    model_base,
    enum_val,    # steered toward this class
    sing_val,
    candidate_directions,
    SELECT_DIR,
    batch_size=4,
)

print(f"Best direction — position index: {pos}, layer: {layer}")
print(f"Direction shape: {best_dir.shape}")

## 5 — Find classification threshold (ROC curve)

Project every val-set hidden state onto the direction → scalar score.
Sweep thresholds and pick the point on the ROC curve closest to (0, 1).

In [ ]:
from pipeline.utils.threshold_utils import get_threshold_by_curve
import json

dir_vector = candidate_directions[pos, layer]   # shape: (d_model,)

fpr, tpr, roc_auc, best_idx, threshold = get_threshold_by_curve(
    dir_vector, model_base, pos, layer,
    sing_val, enum_val,   # (negative, positive) — check arg order in their code
)

print(f"ROC AUC: {roc_auc:.4f}")
print(f"Best threshold: {threshold:.4f}")

# Save metadata
metadata = {
    "pos": int(pos),
    "layer": int(layer),
    "roc_auc": float(roc_auc),
    "threshold": float(threshold),
    "dataset": DATASET_NAME,
}
with open(f"{SELECT_DIR}/direction_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)
print("Metadata saved.")

In [ ]:
# ── Optional: plot ROC curve ─────────────────────────────────────────
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
plt.scatter(fpr[best_idx], tpr[best_idx], color='red', zorder=5,
            label=f"threshold = {threshold:.2f}")
plt.plot([0, 1], [0, 1], 'k--', linewidth=0.8)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"Enumerability direction ROC — layer {layer}")
plt.legend()
plt.tight_layout()
plt.savefig(f"{ARTIFACT_DIR}/roc_curve.png", dpi=150)
plt.show()

## 6 — Evaluate by projection on test set

In [ ]:
from evaluate import evaluate_by_projecting

EVAL_DIR = f"{ARTIFACT_DIR}/eval_test"
os.makedirs(EVAL_DIR, exist_ok=True)

evaluate_by_projecting(
    EVAL_DIR,
    sing_test,   # "answerable" in the paper's API → our singular
    enum_test,   # "unanswerable" → our enumerable
    model_base,
    dir_vector,
    pos, layer, threshold,
)

In [ ]:
# Print results
import json
with open(f"{EVAL_DIR}/evaluation_results.json") as f:
    results = json.load(f)
print(json.dumps(results, indent=2))

## 7 — Activation steering demo

This is where we go beyond the paper: **add** the direction at inference
time and watch the model's completion shift from singular-style to
enumerable-style, and vice versa when we subtract it.

We reuse the paper's `get_activation_addition_input_pre_hook` from
`hook_utils.py` — no changes needed.

In [ ]:
import torch
from pipeline.utils.hook_utils import get_activation_addition_input_pre_hook, add_hooks

def steer_and_generate(question: str, coeff: float, layer: int,
                        dir_vector, model_base, max_new_tokens: int = 80):
    """
    Generate a completion with the direction vector added at `layer`.
    coeff > 0  →  push toward enumerable
    coeff < 0  →  push toward singular
    coeff = 0  →  baseline (no steering)
    """
    coeff_tensor = torch.tensor(coeff, dtype=torch.float32)
    hook_fn = get_activation_addition_input_pre_hook(
        vector=dir_vector.float(),
        coeff=coeff_tensor,
    )
    block = model_base.model_block_modules[layer]

    if coeff == 0:
        completions = model_base.generate_completions(
            [question], fwd_pre_hooks=[], fwd_hooks=[],
            max_new_tokens=max_new_tokens
        )
    else:
        with add_hooks(module_forward_pre_hooks=[(block, hook_fn)],
                       module_forward_hooks=[]):
            completions = model_base.generate_completions(
                [question], fwd_pre_hooks=[], fwd_hooks=[],
                max_new_tokens=max_new_tokens
            )
    return completions[0]['response']

In [ ]:
# ── Example: a question that normally has one answer ──────────────────
q_singular = "Who wrote Romeo and Juliet?"

print("=" * 60)
print(f"Question: {q_singular}")
print()
for coeff, label in [( 0,   "Baseline (no steering)"),
                      ( 15,  "+ direction (push → enumerable)"),
                      (-15,  "− direction (push → singular)")]:
    response = steer_and_generate(q_singular, coeff, layer, dir_vector, model_base)
    print(f"[coeff={coeff:+d}] {label}")
    print(f"  {response}")
    print()

In [ ]:
# ── Example: a question that naturally has multiple answers ───────────
q_enum = "What languages are spoken in Switzerland?"

print("=" * 60)
print(f"Question: {q_enum}")
print()
for coeff, label in [( 0,   "Baseline (no steering)"),
                      ( 15,  "+ direction (push → enumerable)"),
                      (-15,  "− direction (push → singular)")]:
    response = steer_and_generate(q_enum, coeff, layer, dir_vector, model_base)
    print(f"[coeff={coeff:+d}] {label}")
    print(f"  {response}")
    print()

## 8 — Cross-dataset generalisation

Train the direction on one dataset, evaluate on another.
This tests whether the direction is truly capturing a general concept
of enumerability rather than dataset-specific surface features.

In [ ]:
# Only run this after you have directions for all three datasets.
# Set SAVED_DIRS to point at the artifact dirs from step 3.

# Example: evaluate the AmbigQA-trained direction on SituatedQA test set
# Reload direction:
# import torch
# mean_diffs_ambig = torch.load('runs/ambigqa/mean_diffs.pt')
# dir_vec_ambig = mean_diffs_ambig[pos, layer]
#
# evaluate_by_projecting(
#     'runs/ambigqa/eval_on_situatedqa',
#     sing_test_situated, enum_test_situated,
#     model_base, dir_vec_ambig, pos, layer, threshold
# )

print("Cross-dataset eval: uncomment the block above after training on multiple datasets.")

## 9 — Layer sweep (optional diagnostic)

Plot projection accuracy per layer to understand at which depth the
enumerability feature is most linearly accessible.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from pipeline.generate_directions import get_mean_activations

# For a small diagnostic subset (keep it fast)
N_DIAG = 100
diag_enum = enum_val[:N_DIAG]
diag_sing = sing_val[:N_DIAG]

n_layers = model_base.model.config.num_hidden_layers

# Extract hidden states for all layers at once using get_mean_activations
# We need per-sample vectors, not the mean — so we run per-sample here
def extract_per_sample(questions, layer_idx):
    """Extract last-token hidden state at layer_idx for each question."""
    vecs = []
    for q in questions:
        inputs = model_base.tokenize_instructions_fn([q])
        with torch.no_grad():
            out = model_base.model(
                input_ids=inputs.input_ids.to(model_base.model.device),
                attention_mask=inputs.attention_mask.to(model_base.model.device),
                output_hidden_states=True,
            )
        # hidden_states[layer_idx] shape: (1, seq_len, d_model)
        vec = out.hidden_states[layer_idx][0, -1, :].cpu().float().numpy()
        vecs.append(vec)
    return np.array(vecs)

sweep_layers = list(range(0, n_layers, 4))
accs = []

for l in sweep_layers:
    X_enum = extract_per_sample(diag_enum, l)
    X_sing = extract_per_sample(diag_sing, l)
    X = np.concatenate([X_enum, X_sing], axis=0)
    y = np.array([1]*len(X_enum) + [0]*len(X_sing))

    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X)

    probe = LogisticRegression(max_iter=500, random_state=42)
    cv_acc = cross_val_score(probe, X_sc, y, cv=3, scoring='accuracy').mean()
    accs.append(cv_acc)
    print(f"  Layer {l:2d}: acc = {cv_acc:.3f}")

plt.figure(figsize=(8, 4))
plt.plot(sweep_layers, accs, marker='o')
plt.axvline(layer, color='red', linestyle='--', label=f'selected layer {layer}')
plt.xlabel("Layer")
plt.ylabel("3-fold CV accuracy")
plt.title("Enumerability linear probe accuracy per layer")
plt.legend()
plt.tight_layout()
plt.savefig(f"{ARTIFACT_DIR}/layer_sweep.png", dpi=150)
plt.show()